In [0]:
%sql 
-- Which use categories dominate new activity — shifting from single-family toward multi-unit?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_use_category_growth AS
WITH use_cats AS (
    SELECT 
        fp.permit_nbr,
        dd.year,
        CASE 
            WHEN (dut.use_code = 1 AND dut.use_desc = 'Dwelling - Single Family') 
            OR (dut.use_code = 35 AND dut.use_desc = 'Condo-Single Family')
            OR (dut.use_code = 35 AND dut.use_desc = 'Condo-Duplex')
            THEN 'single_family'
            WHEN dut.use_code IN (2, 5) 
            OR (dut.use_code = 35 AND dut.use_desc = 'Condo-Multi Family')
            THEN 'multi_unit'
            WHEN dut.use_desc LIKE '%Accessory Dwelling Unit%' 
            THEN 'adu'
            WHEN dut.use_desc LIKE '%Store%'
            OR dut.use_desc LIKE '%Office%' 
            OR dut.use_desc LIKE '%Restaurant%'
            OR dut.use_desc LIKE '%Shop%'
            THEN 'commercial'
            ELSE 'other' 
        END AS use_category
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_use_type AS dut
        ON fp.use_type_key = dut.use_type_key
    LEFT JOIN la_lakehouse.gold.dim_date AS dd
        ON fp.issue_date_key = dd.date_key
    WHERE dd.year BETWEEN 2021 AND 2026
),
yearly_counts AS (
    SELECT 
        year,
        use_category,
        COUNT(permit_nbr) AS permit_count
    FROM use_cats
    GROUP BY year, use_category
),
market_share_percentage AS (
    SELECT 
        year,
        use_category,
        permit_count,
        SUM(permit_count) OVER (PARTITION BY year) AS annual_total,
        ROUND(100.0 * permit_count / SUM(permit_count) OVER (PARTITION BY year), 2) AS market_share_pect
    FROM yearly_counts
), 
pivot_summary AS (
    SELECT 
        use_category,
        MAX(CASE WHEN year = 2021 THEN permit_count END) AS count_2021,
        MAX(CASE WHEN year = 2025 THEN permit_count END) AS count_2025,
        MAX(CASE WHEN year = 2021 THEN market_share_pect END) AS share_2021, 
        MAX(CASE WHEN year = 2025 THEN market_share_pect END) AS share_2025
    FROM market_share_percentage
    GROUP BY use_category
)
SELECT
    use_category,
    count_2021,
    count_2025,
    share_2021,
    share_2025,
    ROUND(share_2025 - share_2021, 2) AS net_shift_pct
FROM pivot_summary
ORDER BY net_shift_pct DESC;


#Testing The Views

In [0]:
%sql 
SELECT * 
FROM la_lakehouse.gold.vw_use_category_growth;